In [13]:
import os
import warnings
import numpy as np
import ollama
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

In [ ]:
def load_document(file_path: str):
    #Loads a PDF or TXT document and returns its text content.
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    if file_path.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file_path.endswith(".txt"):
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError("Unsupported file format. Please use .pdf or .txt")

    docs = loader.load()
    return docs

In [15]:
def get_doc_embedding(file_path: str, model_name: str = "nomic-embed-text-v2-moe"):
    #Loads document, splits text, creates chunk embeddings, and averages them to represent the document.
    docs = load_document(file_path)

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_documents(docs)

    chunk_texts = [doc.page_content for doc in chunks]

    response = ollama.embed(model=model_name, input=chunk_texts)
    embeddings = np.array(response.embeddings)

    # Average chunk embeddings to produce a single document vector
    doc_vector = np.mean(embeddings, axis=0)
    return doc_vector

In [16]:
def similarity_checks(vec1: np.ndarray, vec2: np.ndarray):
#Calculates similarity metrics between two document vectors.
    dot_sim = np.dot(vec1, vec2)
    cos_sim = dot_sim / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    euclidean_dist = np.linalg.norm(vec1 - vec2)

    print(f"DOT PRODUCT SIM_SCORE: {dot_sim:.4f}")
    print(f"COSINE SIM_SCORE: {cos_sim:.4f}")
    print(f"EUCLIDEAN DISTANCE: {euclidean_dist:.4f}\n")

    return {
        "dot_product": dot_sim,
        "cosine_similarity": cos_sim,
        "euclidean_distance": euclidean_dist,
    }

In [17]:
def compare_documents(doc1_path: str, doc2_path: str):
    """Executes document comparison flow."""
    print(f"Processing Doc 1: {doc1_path}")
    vec1 = get_doc_embedding(doc1_path)

    print(f"Processing Doc 2: {doc2_path}")
    vec2 = get_doc_embedding(doc2_path)

    print("\n--- Document Similarity Results ---")
    return similarity_checks(vec1, vec2)

In [18]:

if __name__ == "__main__":
    doc_path_1 = "data/PROPOSAL RESTAURANT.pdf"
    doc_path_2 = "data/VALGUS_MED_PROPOSAL & AUDIT.pdf"

    compare_documents(doc_path_1, doc_path_2)

Processing Doc 1: data/PROPOSAL RESTAURANT.pdf
Processing Doc 2: data/VALGUS_MED_PROPOSAL & AUDIT.pdf

--- Document Similarity Results ---
DOT PRODUCT SIM_SCORE: 0.4676
COSINE SIM_SCORE: 0.7610
EUCLIDEAN DISTANCE: 0.5420



In [19]:
doc_1_embedding = get_doc_embedding(doc_path_1)
doc_2_embedding = get_doc_embedding(doc_path_2)

In [20]:
print(doc_1_embedding)
print(doc_2_embedding)

[ 3.98553448e-02  3.67914713e-02 -2.37941268e-02 -2.99412077e-02
  2.74166950e-03 -2.77628063e-02 -3.19660118e-02  7.14106243e-03
  6.34727007e-03  3.38229542e-02  1.62447087e-03  1.76404767e-02
 -8.87716443e-03 -8.91837406e-03  1.52833783e-02  4.69196458e-02
 -4.19683210e-02  1.39791889e-02 -1.84080932e-02 -2.02268650e-02
  2.33577468e-02 -1.72775110e-02  2.24305528e-03 -5.22088515e-02
 -7.46078122e-03  2.99271836e-02 -2.97885412e-02  1.46538593e-02
  4.40371222e-02  3.04448020e-02  1.16289620e-02 -2.26243699e-02
 -1.40999369e-02  8.23970927e-02  6.94577317e-04  7.42525287e-02
 -3.26298133e-02 -2.22486712e-02 -2.00663160e-02  2.60953000e-02
  4.14404623e-02  4.59616147e-02  1.69019558e-03 -3.54851890e-03
 -2.88994347e-02  5.41577480e-02  2.73377858e-02  1.85777583e-02
 -3.83804590e-02  3.40582260e-02 -2.11838413e-02 -1.56640285e-02
  3.91268122e-02  1.09242818e-02  5.54329852e-03 -2.35809441e-02
 -1.28099140e-02 -1.17707152e-02  7.31658720e-02  2.10650824e-02
 -5.73519217e-04  7.97561

In [21]:
similarity_checks(doc_1_embedding, doc_2_embedding)

DOT PRODUCT SIM_SCORE: 0.4676
COSINE SIM_SCORE: 0.7610
EUCLIDEAN DISTANCE: 0.5420



{'dot_product': np.float64(0.4676129555042811),
 'cosine_similarity': np.float64(0.761037426038524),
 'euclidean_distance': np.float64(0.5419930411039734)}